# Exploratory Data Analysis — Customer Dataset


## 1 — Load Dataset

In [3]:
import pandas as pd
import numpy as np

RAW_PATH = '../../../data/customers_clean.csv'

df = pd.read_csv(RAW_PATH)
print(f'Shape: {df.shape}')
df.head()

Shape: (5504, 15)


,customer_id,age,city,customer_segment,product_category,order_count,average_order_value,total_spend,discount_percentage,days_since_last_order,website_visits,support_tickets,return_count,payment_type,customer_tenure_days
0,1,58.0,Chennai,Premium,Sports,18,12622.15,227198.70,6.98,53,174,17,2,Cash,894
1,2,20.0,Chennai,Premium,Clothing,15,25515.09,382726.35,1.33,102,184,20,13,Debit Card,949
2,3,55.0,Delhi,Premium,Books,11,35057.90,385636.90,17.01,80,56,10,1,Credit Card,808
3,4,24.0,Bangalore,Regular,Furniture,17,40452.85,687698.45,36.49,275,32,12,2,Net Banking,630
4,5,58.0,Bangalore,Inactive,Clothing,46,3942.98,181377.08,33.06,149,21,7,6,Upi,599


In [4]:
# Numeric columns only — univariate analysis applies to these
numeric_cols = df.select_dtypes(include='number').columns.tolist()
print('Numeric columns:')
print(numeric_cols)

Numeric columns:
['customer_id', 'age', 'order_count', 'average_order_value', 'total_spend', 'discount_percentage', 'days_since_last_order', 'website_visits', 'support_tickets', 'return_count', 'customer_tenure_days']


## 2 — Univariate Analysis

Univariate analysis examines each column independently.
We focus on three things:
- **Skewness** — is the distribution symmetric or pulled to one side?
- **Outliers via IQR** — values that fall far outside the middle 50% of the data
- **Outliers via Z-Score** — values that are more than 3 standard deviations from the mean

## 2a — Skewness Assessment

In [5]:
skewness = df[numeric_cols].skew().sort_values(ascending=False)

print(f'{"Column":<30} {"Skewness":>10}  Interpretation')
print('-' * 65)
for col, skew in skewness.items():
    if skew > 1:
        label = 'Right-skewed (positive)'
    elif skew < -1:
        label = 'Left-skewed (negative)'
    else:
        label = 'Approximately symmetric'
    print(f'{col:<30} {skew:>10.4f}  {label}')

Column                           Skewness  Interpretation
-----------------------------------------------------------------
total_spend                        1.1194  Right-skewed (positive)
return_count                       0.9608  Approximately symmetric
discount_percentage                0.4827  Approximately symmetric
website_visits                     0.4790  Approximately symmetric
customer_tenure_days               0.0240  Approximately symmetric
customer_id                        0.0001  Approximately symmetric
days_since_last_order              0.0000  Approximately symmetric
order_count                       -0.0222  Approximately symmetric
average_order_value               -0.0250  Approximately symmetric
age                               -0.0347  Approximately symmetric
support_tickets                   -0.0359  Approximately symmetric


## 2b — Outlier Detection: IQR Method



In [6]:
print(f'{"Column":<30} {"Q1":>10} {"Q3":>10} {"IQR":>10} {"Lower":>12} {"Upper":>12} {"Outliers":>10} {"% of col":>10}')
print('-' * 105)

iqr_summary = {}

for col in numeric_cols:
    col_data = df[col].dropna()
    Q1 = col_data.quantile(0.25)
    Q3 = col_data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_mask = (col_data < lower) | (col_data > upper)
    n_outliers = outlier_mask.sum()
    pct = (n_outliers / len(col_data)) * 100
    iqr_summary[col] = n_outliers
    print(f'{col:<30} {Q1:>10.2f} {Q3:>10.2f} {IQR:>10.2f} {lower:>12.2f} {upper:>12.2f} {n_outliers:>10} {pct:>9.2f}%')

Column                                 Q1         Q3        IQR        Lower        Upper   Outliers   % of col
---------------------------------------------------------------------------------------------------------
customer_id                       1376.75    4125.25    2748.50     -2746.00      8248.00          0      0.00%
age                                 32.00      57.00      25.00        -5.50        94.50          0      0.00%
order_count                         13.00      38.00      25.00       -24.50        75.50          0      0.00%
average_order_value              13141.27   37536.88   24395.60    -23452.13     74130.28          0      0.00%
total_spend                     184261.43 1015455.94  831194.51  -1062530.34   2262247.71         57      1.04%
discount_percentage                 12.87      37.39      24.52       -23.92        74.17         25      0.45%
days_since_last_order               92.00     275.00     183.00      -182.50       549.50          0      0.00

## 2c — Outlier Detection: Z-Score Method


In [7]:
print(f'{"Column":<30} {"Mean":>12} {"Std":>12} {"Outliers (|Z|>3)":>18} {"% of col":>10}')
print('-' * 90)

zscore_summary = {}

for col in numeric_cols:
    col_data = df[col].dropna()
    mean = col_data.mean()
    std = col_data.std()
    if std == 0:
        print(f'{col:<30} std=0, skipping')
        continue
    z_scores = (col_data - mean) / std
    n_outliers = (z_scores.abs() > 3).sum()
    pct = (n_outliers / len(col_data)) * 100
    zscore_summary[col] = n_outliers
    print(f'{col:<30} {mean:>12.2f} {std:>12.2f} {n_outliers:>18} {pct:>9.2f}%')

Column                                 Mean          Std   Outliers (|Z|>3)   % of col
------------------------------------------------------------------------------------------
customer_id                         2750.50      1587.52                  0      0.00%
age                                   44.33        15.04                  0      0.00%
order_count                           25.71        14.51                  0      0.00%
average_order_value                25377.71     14180.25                  0      0.00%
total_spend                       667172.45    588976.02                 15      0.27%
discount_percentage                   25.31        15.15                 25      0.45%
days_since_last_order                183.35       106.12                  0      0.00%
website_visits                       102.10        60.22                 15      0.27%
support_tickets                       10.16         6.04                  0      0.00%
return_count                          1